# Shortlister training (Role A, version 2: the LeetCode-style tutor)

QLoRA fine-tuning of the misconception shortlister as a **bi-encoder**: the query from
`build_query()` and every Eedi misconception name are embedded by the same model, and
misconceptions are ranked by cosine similarity.

**Its job in the tutor** (docs/design_decisions.md, "What we are trying to build"): when no
rule reproduces a student's step, the shortlister ranks the in-topic Eedi names and the LLM
fallback picks one from its top 25, or none. So this notebook optimises **Recall@25 on
live-format calculation steps**, the queries the tutor actually sends.

**Kaggle setup**
1. Accelerator: **GPU T4 x1** (or P100). Internet: **on**.
2. *Add Input* -> the competition **eedi-mining-misconceptions-in-mathematics**.
3. *Add Input* -> your private dataset holding `adapter-3B-llm/` (run 3, the baseline) and
   `llm_examples.csv`. Both are found wherever they are mounted.
4. Our code: *Add-ons -> Secrets* -> `GITHUB_TOKEN` with read access to Zenith053/MathEasy
   (or a private dataset **matheasy-code** holding the repo).

**What one Save & Run All does**
1. Scores run 3 as it is (the baseline) -> `/kaggle/working/scores_run3.json`.
2. Trains the new adapter -> `/kaggle/working/adapter-3B-v2/`.
3. Scores the new adapter the same way -> `/kaggle/working/scores_v2.json`, and prints both side by side.

Scores come from `matheasy/shortlister/score_baseline.py`: val and test calculation rows,
all 2,587 names and the 972 in-topic names, training and live format. Only scores are
written to the output, never Eedi text.

`SMOKE = True` is a quick check (100 training pairs, 1 epoch). Set `SMOKE = False` for the real run.
The switches in the config cell reproduce the earlier runs: run 3 is `USE_LIVE = CALC_ONLY =
IN_TOPIC = False`, `EPOCHS = 3`, `BATCH = 8`, `MAX_Q = 512`, `SELECT_BY = "MAP@25"`.

In [ ]:
!pip install -q -U peft bitsandbytes lark rank_bm25

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/tmp/MathEasy")  # not /kaggle/working: Eedi-derived files must not end up in the saved output
if not WORK.exists():
    code_dir = next(INPUT.rglob("prepare_eedi.py"), None)  # matheasy-code dataset, wherever it is mounted
    if code_dir:
        # The package folder is the one above shortlister/; Kaggle may or may not keep "matheasy/" in the path.
        shutil.copytree(code_dir.parents[1], WORK / "matheasy")
        # B's rules read eval/held_back_rules.json, and rule_pairs.py uses eval/build_probes.py.
        shutil.copytree(next(INPUT.rglob("held_back_rules.json")).parent, WORK / "eval")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://{token}@github.com/Zenith053/MathEasy.git", str(WORK)], check=True)

# The scripts read Eedi from data/eedi/raw: point it at the competition input.
eedi = next(INPUT.rglob("misconception_mapping.csv")).parent
raw = WORK / "data" / "eedi" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    raw.symlink_to(eedi)
print("code:", WORK, " eedi:", eedi)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
!python -m matheasy.shortlister.prepare_eedi
!python -m matheasy.shortlister.query
!python -m matheasy.shortlister.live_queries
!python -m matheasy.shortlister.hard_negatives

In [ ]:
# 1. The baseline: run 3, scored as the tutor uses it.
import torch
RUN3 = next((p.parent for p in INPUT.rglob("adapter_config.json")), None)  # the one adapter attached: run 3
if RUN3 is None:
    print("no adapter attached: no baseline")
else:
    !python -m matheasy.shortlister.score_baseline --adapter "{RUN3}" --out /kaggle/working/scores_run3.json

## 2. Train the new adapter

In [ ]:
SMOKE = True
SIZE = "3B"                          # "1.5B" scored lower (val Recall@10 0.572 against 0.655)
USE_LLM = True                       # add the LLM-written examples (attach llm_examples.csv)
USE_RULES = False                    # rule pairs: a step a rule explains never reaches the shortlister
USE_LIVE = True                      # add the live-format copy of each training row that has one
CALC_ONLY = True                     # calculation questions only (prepare_eedi.py's Form)
IN_TOPIC = True                      # rank, and draw negatives from, the 972 in-topic names (candidates.py)
SELECT_BY = "Recall@25"              # on val, live and training format together

MODEL = f"Qwen/Qwen2.5-{SIZE}-Instruct"
N_TRAIN = 100 if SMOKE else None     # None = all training pairs
EPOCHS = 1 if SMOKE else 6           # every earlier run was still improving at its last epoch (3)
BATCH = 16                           # in-batch negatives: bigger is better if memory allows
LR = 1e-4
TEMP = 0.05
N_HARD = 3                           # hard negatives per pair (0 = in-batch negatives only)
MAX_Q, MAX_M = 256, 96               # token limits; the longest query is ~200 tokens
LORA_R = 16
SEED = 20261016
WARMUP = 0.1                         # share of steps with rising learning rate
OUT = Path(f"/kaggle/working/adapter-{SIZE}-v2" + ("-smoke" if SMOKE else ""))

INSTRUCT = "Instruct: Given a student's wrong step, retrieve the misconception that explains it\nQuery: "

In [ ]:
import pandas as pd
from matheasy.shortlister.candidates import candidate_ids
from matheasy.shortlister.hard_negatives import mine
from matheasy.shortlister.shortlist import map_at_k, recall_at_k

KEY = ["QuestionId", "WrongAnswer"]
pairs = pd.read_csv("data/eedi/pairs.csv")
names = pd.read_csv("data/eedi/raw/misconception_mapping.csv")
names["MisconceptionName"] = names.MisconceptionName.str.strip()
if IN_TOPIC:
    names = names[names.MisconceptionId.isin(candidate_ids())].reset_index(drop=True)
name_of = dict(zip(names.MisconceptionId, names.MisconceptionName))


def load(path):
    rows = pd.read_csv(path)[KEY + ["Split", "MisconceptionId", "Query"]].merge(pairs[KEY + ["Form"]], on=KEY)
    return rows[rows.Form == "calculation"] if CALC_ONLY else rows


queries, live = load("data/eedi/queries.csv"), load("data/eedi/live_queries.csv")
train = queries[queries.Split == "train"]
if USE_LIVE:
    train = pd.concat([train, live[live.Split == "train"]])   # same keys, so the same negatives
train = train[train.MisconceptionId.isin(name_of)].sample(frac=1, random_state=SEED)
if N_TRAIN:
    train = train.head(N_TRAIN)
val = {"training format": queries[queries.Split == "val"], "live format": live[live.Split == "val"]}
test = {"training format": queries[queries.Split == "test"], "live format": live[live.Split == "test"]}
seen = set(train.MisconceptionId)
print(len(train), "train pairs;", {k: len(v) for k, v in val.items()}, "val;", {k: len(v) for k, v in test.items()},
      "test;", len(names), "candidate misconceptions")

# Hard negatives, mined among the candidate names: siblings (other wrong options of the same
# question) and look-alike names.
negs_of = {}


def add_negatives(rows):
    for r in mine(rows[KEY + ["MisconceptionId"]].drop_duplicates(KEY), names).itertuples():
        negs_of[(r.QuestionId, r.WrongAnswer)] = [int(m) for m in str(r.NegativeIds).split()][:N_HARD]


add_negatives(train)


def add_extra(train, extra, tag):
    # Each extra row is its own pseudo-question (no siblings), so its negatives are look-alike names.
    # Keys are (pseudo-question, tag), so LLM and rule rows never clash.
    extra = extra[extra.MisconceptionId.isin(name_of)].reset_index(drop=True)
    extra = extra.assign(QuestionId=-1 - extra.index, WrongAnswer=tag)
    add_negatives(extra)
    print(f"+ {len(extra)} {tag} pairs")
    return pd.concat([train, extra[KEY + ["MisconceptionId", "Query"]]])


if USE_LLM:
    llm = pd.read_csv(next(INPUT.rglob("llm_examples.csv")))
    train = add_extra(train, llm[llm.Topic != "other"].dropna(subset=["Query"]), "llm")
if USE_RULES:
    !python -m matheasy.shortlister.rule_pairs
    train = add_extra(train, pd.read_csv("data/eedi/rule_pairs.csv"), "rule")
train = train.sample(frac=1, random_state=SEED)
print(len(train), "training pairs")

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "right"
model = AutoModel.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_use_double_quant=True),
    dtype=torch.float16, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()


def embed(texts, max_len):
    """Last-token pooling, L2-normalised."""
    batch = tok([t + tok.eos_token for t in texts], max_length=max_len, truncation=True,
                padding=True, return_tensors="pt").to(model.device)
    hidden = model(**batch).last_hidden_state
    last = batch["attention_mask"].sum(1) - 1
    return F.normalize(hidden[torch.arange(len(texts)), last].float(), dim=-1)


@torch.no_grad()
def evaluate(df, k=50):
    model.eval()
    m = torch.cat([embed(names.MisconceptionName[i:i + 64].tolist(), MAX_M)
                   for i in range(0, len(names), 64)])
    q = torch.cat([embed([INSTRUCT + t for t in df.Query[i:i + 16]], MAX_Q)
                   for i in range(0, len(df), 16)])
    top = (q @ m.T).topk(k, dim=1).indices.cpu()
    ranked = [[names.MisconceptionId.iloc[j] for j in row] for row in top.tolist()]
    true = df.MisconceptionId.tolist()
    model.train()
    unseen = [t not in seen for t in true]
    pick = lambda keep: ([r for r, u in zip(ranked, unseen) if u == keep], [t for t, u in zip(true, unseen) if u == keep])
    return {"n": len(true), "Recall@10": recall_at_k(ranked, true, 10), "Recall@25": recall_at_k(ranked, true, 25),
            "Recall@50": recall_at_k(ranked, true, 50), "MAP@25": map_at_k(ranked, true, 25),
            "Recall@25 seen": recall_at_k(*pick(False), 25), "Recall@25 unseen": recall_at_k(*pick(True), 25)}


def combined(results):
    # The selection score: SELECT_BY over val, each format weighted by its number of rows.
    n = sum(r["n"] for r in results.values())
    return sum(r[SELECT_BY] * r["n"] for r in results.values()) / n


show = lambda r: "  ".join(f"{name} {v:.3f}" for name, v in r.items() if name != "n")

In [ ]:
import time
from transformers import get_linear_schedule_with_warmup

for name, rows in val.items():
    print(f"before training, val {name}: ", show(evaluate(rows)))

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = EPOCHS * (len(train) // BATCH)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
scaler = torch.amp.GradScaler()
best = None
model.train()
start = time.time()
for epoch in range(EPOCHS):
    shuffled = train.sample(frac=1, random_state=SEED + epoch)
    for i in range(0, len(shuffled) - BATCH + 1, BATCH):
        b = shuffled.iloc[i:i + BATCH]
        # Candidates: the batch's true misconceptions, then each pair's hard negatives.
        cand = b.MisconceptionId.tolist() + [m for key in zip(b.QuestionId, b.WrongAnswer)
                                             for m in negs_of.get(key, [])]
        with torch.autocast("cuda", dtype=torch.float16):
            q = embed([INSTRUCT + t for t in b.Query], MAX_Q)
            p = embed([name_of[m] for m in cand], MAX_M)
        logits = (q @ p.T) / TEMP
        # A candidate with the row's own misconception is not a negative (except its own column).
        pos = torch.tensor(b.MisconceptionId.values, device=logits.device)
        same = pos[:, None] == torch.tensor(cand, device=logits.device)[None, :]
        same[torch.arange(len(b)), torch.arange(len(b))] = False
        loss = F.cross_entropy(logits.masked_fill(same, -1e4), torch.arange(len(b), device=logits.device))
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if (i // BATCH) % 5 == 0:
            print(f"epoch {epoch} step {i // BATCH:4d}  loss {loss.item():.3f}  {time.time() - start:.0f}s")
    v = {name: evaluate(rows) for name, rows in val.items()}
    for name, r in v.items():
        print(f"epoch {epoch}, val {name}: ", show(r))
    print(f"epoch {epoch}, val {SELECT_BY} (both formats): {combined(v):.3f}")
    if best is None or combined(v) > combined(best["val"]):
        # Keep this epoch's adapter; test is only read for the epoch chosen on val.
        best = {"epoch": epoch, "val": v, "test": {name: evaluate(rows) for name, rows in test.items()}}
        model.save_pretrained(OUT)
        tok.save_pretrained(OUT)

In [ ]:
# 3. The new adapter, scored exactly like the baseline, then both side by side.
import json, gc
print(f"best epoch {best['epoch']} ({SIZE}, USE_LLM={USE_LLM}, USE_LIVE={USE_LIVE}, CALC_ONLY={CALC_ONLY}, "
      f"IN_TOPIC={IN_TOPIC}, SELECT_BY={SELECT_BY})")
for split in ("val", "test"):
    for name, r in best[split].items():
        print(f"  {split} {name}: ", show(r))
# Scores only (no Eedi text), so they are safe to keep in the output.
settings = {"model": MODEL, "n_hard": N_HARD, "epochs": EPOCHS, "batch": BATCH, "max_q": MAX_Q, "use_llm": USE_LLM,
            "use_rules": USE_RULES, "use_live": USE_LIVE, "calc_only": CALC_ONLY, "in_topic": IN_TOPIC,
            "select_by": SELECT_BY, "smoke": SMOKE}
(OUT / "results.json").write_text(json.dumps({**settings, **best}, indent=1))
print("saved adapter ->", OUT, sorted(p.name for p in OUT.iterdir()))

del model, opt
gc.collect()
torch.cuda.empty_cache()
!python -m matheasy.shortlister.score_baseline --adapter "{OUT}" --out /kaggle/working/scores_v2.json

runs = {name: Path(f"/kaggle/working/scores_{name}.json") for name in ("run3", "v2")}
runs = {name: json.loads(p.read_text()) for name, p in runs.items() if p.exists()}
print(f"\n{'live-format rows':42}" + "".join(f"{n:>22}" for n in runs))
for cell in next(iter(runs.values()))["live-format rows"]:
    print(f"{cell:42}" + "".join(f"{'R@10 %.2f R@25 %.2f' % (r['live-format rows'][cell]['Recall@10'], r['live-format rows'][cell]['Recall@25']):>22}"
                                 for r in runs.values()))

**Smoke test passes when:** both score files are written, the loss goes down, the evaluations print, and the adapter folder with `results.json` is saved. With only 100 pairs the new scores are not meaningful; the real run is.